In [ ]:
!pip install -q cairosvg datasets huggingface-hub Pillow tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/liya_diplomCC')

DRIVE_ROOT = '/content/drive/MyDrive/liya_diplomCC'
SVG_DIR = f'{DRIVE_ROOT}/data/raw_svg'
PNG_DIR = f'{DRIVE_ROOT}/data/png_512'


In [ ]:
# Search huggingface.co/datasets for "svg logo" to find the best available dataset.
# Recommended IDs to try (verify availability before running):
#   "logo-wizard/modern-logo-dataset"
#   "svgcoder/svg-logos" (if available)
# If no SVG dataset is found, use PNG logos and skip the SVG filter step.

from datasets import load_dataset

DATASET_ID = "logo-wizard/modern-logo-dataset"  # UPDATE after checking HuggingFace
ds = load_dataset(DATASET_ID, split="train")
print(f"Loaded {len(ds)} samples")
print("Features:", ds.features)


In [ ]:
from pathlib import Path
from tqdm import tqdm

Path(SVG_DIR).mkdir(parents=True, exist_ok=True)

for i, item in enumerate(tqdm(ds)):
    if "svg" in item and item["svg"]:
        out_path = f"{SVG_DIR}/{i:06d}.svg"
        with open(out_path, "w") as f:
            f.write(item["svg"])
    elif "image" in item:
        out_path = f"{SVG_DIR}/{i:06d}.png"
        item["image"].save(out_path)

print(f"Saved {i+1} files to {SVG_DIR}")


In [ ]:
from scripts.svg_to_png import batch_convert

stats = batch_convert(SVG_DIR, PNG_DIR, size=512)
print(f"Converted: {stats['success']} OK, {stats['failed']} failed of {stats['total']}")


In [ ]:
from scripts.filter_dataset import filter_dataset
import json

filtered = filter_dataset(PNG_DIR, SVG_DIR, min_paths=3, max_paths=500)
print(f"After filtering: {len(filtered)} valid pairs")

with open(f'{DRIVE_ROOT}/data/filtered_pairs.jsonl', 'w') as f:
    for item in filtered:
        f.write(json.dumps(item) + '\n')
print("Saved data/filtered_pairs.jsonl")


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

sample = random.sample(filtered, min(5, len(filtered)))
fig, axes = plt.subplots(1, len(sample), figsize=(15, 3))
if len(sample) == 1:
    axes = [axes]
for ax, item in zip(axes, sample):
    ax.imshow(Image.open(item['png_path']))
    ax.axis('off')
plt.suptitle(f"5 random logos from {len(filtered)} filtered pairs")
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/experiments/dataset_sample.png', dpi=150)
plt.show()
